In [ ]:
import numpy as np
import rasterio as rio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path

In [ ]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FEATURE_DIR = RESULTS / "features"

In [7]:
country_raster_1km_  = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')

In [6]:
cropland_2019_ = rio.open(DATA / 'cropland_glad.potapov.et.al_p_1km_s_20190101_20191231_go_epsg.4326_v20240624.tif')

In [8]:
cropland_2019 = cropland_2019_.read(1)

In [9]:
cropland_2019 = np.where(cropland_2019 == 255, 0, cropland_2019)

In [10]:
cropland_2019_1km = np.empty(country_raster_1km_.shape, dtype=np.uint8)
reproject(
    source=cropland_2019,
    destination=cropland_2019_1km,
    src_transform=cropland_2019_.transform,
    src_crs=cropland_2019_.crs,
    dst_transform=country_raster_1km_.transform,
    dst_crs=country_raster_1km_.crs,
    resampling=Resampling.nearest,
)

(array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], shape=(16922, 33844), dtype=uint8),
 Affine(1000.0, 0.0, -16922000.0,
        0.0, -1000.0, 8461000.0))

In [11]:
with rio.open(
    FEATURE_DIR / 'cropland-2019-1km.tif',
    "w",
    driver="GTiff",
    height=cropland_2019_1km.shape[0],
    width=cropland_2019_1km.shape[1],
    count=1,
    dtype=cropland_2019_1km.dtype,
    crs=country_raster_1km_.crs,
    transform=country_raster_1km_.transform,
    compress='lzw'
) as dst:
    dst.write(cropland_2019_1km, 1)